In [ ]:
## VAE  (You can also choose other models such as VQVAE and MHVQVAE.)
import torch
from model.VAEs import IVAE
from scripts.train import train_IVAE
from probfun.fitness import rastrigin as fitness_function
from scripts.eval import evaluate,  set_seed



set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = 2
latent_dim = 8
batch_size = 1024
epochs = 500
learning_rate = 0.001
temperature = 10.0  


model = IVAE(input_dim, latent_dim, use_tanh=True, tanh_alpha=1/temperature).to(device)

model , loss_trace = train_IVAE(model=model, device="cuda", input_dim=input_dim, batch_size=batch_size, fitness_function=fitness_function,
                           epochs=epochs,learning_rate =learning_rate,
                            objective = "maximize",
                            blackbox_weight=0.05,
                            blackbox_step=1e-3,
                            blackbox_max_directions=16,
                            blackbox_start_fraction=0.8
                                  )


with torch.no_grad():
     z = torch.randn(1000, input_dim).to(device)
     x_0, *rest= model(z)
     
result = evaluate(model, device=device, input_dim=input_dim,obj=fitness_function, obj_name="rastrigin",maximize=True)

print(result)

In [ ]:
## VQVAE  (You can also choose other models such as VQVAE and MHVQVAE.)
import torch

from model.VAEs import VQVAE
from probfun.fitness import rastrigin as fitness_function
from scripts.eval import set_seed
from scripts.train import train_VQVAE
from scripts.eval import evaluate,  set_seed
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable")

device = torch.device("cuda:0")
set_seed(42)



input_dim = 2
batch_size = 1024
epochs = 500

num_embeddings = 16
embedding_dim = 8


model = VQVAE(
    input_dim=input_dim,
    num_embeddings=num_embeddings,
    embedding_dim=embedding_dim,
    use_tanh=True,
    tanh_scale=4.0,
    tanh_alpha=2.0,
).to(device)

model, loss_trace = train_VQVAE(
    model=model,
    device=device,
    input_dim=input_dim,
    batch_size=batch_size,
    fitness_function=fitness_function,
    objective="maximize",
    blackbox_weight=0.05,
    blackbox_step=1e-3,
    blackbox_max_directions=16,
    blackbox_start_fraction=0.8,
    global_radius=4.0,
    epochs=epochs,
    code_calibration_steps=500,
    routing_calibration_steps=1000,
)


model.eval()
result = evaluate(model, device=device, input_dim=input_dim,obj=fitness_function, obj_name="rastrigin",maximize=True)
print(result)


In [ ]:
## MHVQVAE  (You can also choose other models such as VQVAE and MHVQVAE.)
import torch

from model.VAEs import MHVQVAE
from probfun.fitness import rastrigin as fitness_function
from scripts.eval import evaluate_muti_peak, set_seed
from scripts.train import train_MHVQVAE

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable")

device = torch.device("cuda:0")
set_seed(42)

input_dim = 2
batch_size = 1024
epochs = 500

num_embeddings = 16
embedding_dim = 8
n_heads = 4

model = MHVQVAE(
    input_dim=input_dim,
    num_embeddings=num_embeddings,
    embedding_dim=embedding_dim,
    n_heads=n_heads,
    use_tanh=True,
    tanh_scale=4.0,
    tanh_alpha=2.0,
).to(device)

model, loss_trace = train_MHVQVAE(
    model=model,
    device=device,
    input_dim=input_dim,
    batch_size=batch_size,
    fitness_function=fitness_function,
    objective="maximize",
    global_radius=4.0,
    epochs=epochs,
    head_min_distance=2.0,
    routing_codes=6,
    code_calibration_steps=500,
    routing_calibration_steps=1000,
    blackbox_weight=0.05,
    blackbox_step=1e-3,
    blackbox_max_directions=16,
    blackbox_start_fraction=0.8,
)

print("head targets:", model.head_targets)
print("head objective values:", model.head_target_values)

model.eval()
result = evaluate_muti_peak(
    model.to("cpu"),  # evaluate_muti_peak builds inputs on CPU
    input_dim=input_dim,
    obj=fitness_function,
    obj_name="rastrigin",
    maximize=True,
)
print(result)